In [ ]:
from vtk.numpy_interface import dataset_adapter as dsa
from uvisbox.Modules.SquidGlyphs import squid_glyph_3D 
from uvisbox.Modules.ConeGlyphs import cone_glyph
import numpy as np
import pyvista as pv

def load_isabel_data():
    data = np.load("../uvisbox/Datasets/hurricane_isabel_ensemble_vectors_time10.npz")
    positions = data['positions']
    vectors = data['vectors']
    magnitudes = np.linalg.norm(vectors, axis=2)
    median_magnitudes = np.median(magnitudes, axis=1)
    return positions, vectors, median_magnitudes

# Load data
positions, ensemble_vectors, median_magnitudes = load_isabel_data()

# Visualization parameters
scaling_factor = 0.2
percenti_value = 95

plotter = pv.Plotter(shape=(1, 2), window_size=[6400, 3200])
plotter.subplot(0, 0)
plotter, points, triangles = cone_glyph(positions, ensemble_vectors, point_values=median_magnitudes, 
                                        percentile=percenti_value, scale=scaling_factor, ax=plotter)

plotter.subplot(0, 1)
plotter, points, triangles = squid_glyph_3D(positions, ensemble_vectors, point_values=median_magnitudes, 
                                            percentile=percenti_value, scale=scaling_factor, ax=plotter)
plotter.link_views()
plotter.show()
# plotter.screenshot("hurricane_isabel_cone_squid_glyphs2.png")




In [ ]:
# Plot ensemble vectors - all members using PyVista arrows
num_points = positions.shape[0]
num_ensembles = ensemble_vectors.shape[1]

# Replicate positions for each ensemble member
all_positions = np.repeat(positions, num_ensembles, axis=0)

# Flatten ensemble vectors to match positions
all_vectors = ensemble_vectors.reshape(-1, 3)

# Calculate magnitudes for all vectors
all_magnitudes = np.linalg.norm(all_vectors, axis=1)

plotter = pv.Plotter(window_size=[3200, 3200])
point_cloud = pv.PolyData(all_positions)
point_cloud['vectors'] = all_vectors
point_cloud['magnitude'] = all_magnitudes

arrows = point_cloud.glyph(orient='vectors', scale='magnitude', 
                            factor=0.3, geom=pv.Arrow())
plotter.add_mesh(arrows, scalars='magnitude', cmap='RdBu_r',
                show_scalar_bar=True, opacity=0.5)
plotter.add_title(f'All Ensemble Vectors ({num_ensembles} members per point)')
plotter.show()

print(f"Total arrows plotted: {num_points * num_ensembles}")
